# Neuronové sítě od nuly 🧠

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/navidofek-cmyk/ml_nn_sets/blob/main/colab_tutorial.ipynb)
[![GitHub](https://img.shields.io/badge/GitHub-ml__nn__sets-black)](https://github.com/navidofek-cmyk/ml_nn_sets)
[![Teorie](https://img.shields.io/badge/Teorie-web-6366f1)](https://navidofek-cmyk.github.io/ml_nn_sets)

Tento notebook obsahuje **3 příklady** s předtrénovanými váhami — stačí spustit buňky, trénovat nemusíš.

📖 **Kompletní teorie** (matematika, backprop, Adam...): [navidofek-cmyk.github.io/ml_nn_sets](https://navidofek-cmyk.github.io/ml_nn_sets)

| Část | Úloha | Model |
|------|-------|-------|
| A | Regrese sin(2πx) | 1→32→32→1, ReLU, MSE |
| B | Two-moons klasifikace | 2→64→64→1, Sigmoid, BCE |
| C | MNIST číslic 0–9 | 784→256→128→10, Softmax, CE |

In [ ]:
# Stažení vah z GitHubu (spusť jednou)
import subprocess
BASE = 'https://raw.githubusercontent.com/navidofek-cmyk/ml_nn_sets/main/weights/'
files = ['regression_pt.pth', 'moons_pt.pth', 'moons_scaler_mean.npy',
         'moons_scaler_std.npy', 'mnist_pt.pth']
for f in files:
    subprocess.run(['wget', '-q', '-nc', BASE + f], check=True)
    print(f'  ✓ {f}')
print('Hotovo!')

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
print(f'PyTorch {torch.__version__}')

---
## Část A: Regrese — sin(2πx)

In [ ]:
# Definice sítě (musí odpovídat architektuře při tréninku)
model_reg = nn.Sequential(
    nn.Linear(1, 32), nn.ReLU(),
    nn.Linear(32, 32), nn.ReLU(),
    nn.Linear(32, 1)
)

# Načtení přetrénovaných vah
model_reg.load_state_dict(torch.load('regression_pt.pth', map_location='cpu'))
model_reg.eval()
print('Váhy načteny ✓')

In [ ]:
# Vizualizace
x_plot = torch.linspace(-1, 1, 300).reshape(-1, 1)
with torch.no_grad():
    y_pred = model_reg(x_plot).numpy()

rng = np.random.default_rng(99)
x_data = rng.uniform(-1, 1, 200)
y_data = np.sin(2 * np.pi * x_data) + rng.normal(0, 0.1, 200)

plt.figure(figsize=(10, 4))
plt.scatter(x_data, y_data, s=12, alpha=0.5, label='Testovací data')
plt.plot(x_plot.numpy(), y_pred, 'r-', lw=2, label='Predikce NN')
plt.plot(x_plot.numpy(), np.sin(2*np.pi*x_plot.numpy()), 'g--', lw=1.5, label='sin(2πx)')
plt.legend(); plt.title('Regrese: síť naučená aproximovat sin(2πx)'); plt.show()

# Interaktivní predikce
x_test = 0.25
pred = model_reg(torch.tensor([[x_test]], dtype=torch.float32)).item()
true = np.sin(2 * np.pi * x_test)
print(f'x = {x_test}  →  predikce = {pred:.4f},  skutečnost = {true:.4f}')

---
## Část B: Binární klasifikace — Two-moons

In [ ]:
model_moon = nn.Sequential(
    nn.Linear(2, 64), nn.ReLU(),
    nn.Linear(64, 64), nn.ReLU(),
    nn.Linear(64, 1), nn.Sigmoid()
)
model_moon.load_state_dict(torch.load('moons_pt.pth', map_location='cpu'))
model_moon.eval()

# Načti parametry normalizace
scaler_mean  = np.load('moons_scaler_mean.npy')
scaler_scale = np.load('moons_scaler_std.npy')

def predict_moon(x1, x2):
    x = np.array([[x1, x2]], dtype=np.float32)
    x_norm = (x - scaler_mean) / scaler_scale
    with torch.no_grad():
        p = model_moon(torch.tensor(x_norm)).item()
    return p

print('Váhy načteny ✓')
print(f'Bod (1.0, 0.0)  → P(třída 1) = {predict_moon(1.0, 0.0):.3f}')
print(f'Bod (-0.5, 0.5) → P(třída 1) = {predict_moon(-0.5, 0.5):.3f}')

In [ ]:
# Vizualizace rozhodovací hranice
from sklearn.datasets import make_moons
X, y = make_moons(n_samples=400, noise=0.15, random_state=42)

h = 0.05
xx, yy = np.meshgrid(np.arange(-2.5, 3.5, h), np.arange(-2, 2.5, h))
grid = np.c_[xx.ravel(), yy.ravel()].astype(np.float32)
grid_norm = (grid - scaler_mean) / scaler_scale
with torch.no_grad():
    Z = model_moon(torch.tensor(grid_norm)).numpy().reshape(xx.shape)

plt.figure(figsize=(8, 5))
plt.contourf(xx, yy, Z, levels=50, cmap='RdBu', alpha=0.7)
plt.scatter(X[y==0,0], X[y==0,1], s=15, c='blue', alpha=0.6, label='Třída 0')
plt.scatter(X[y==1,0], X[y==1,1], s=15, c='red',  alpha=0.6, label='Třída 1')
plt.colorbar(label='P(třída 1)')
plt.legend(); plt.title('Rozhodovací hranice NN — Two-moons'); plt.show()

---
## Část C: MNIST — rozpoznávání číslic

In [ ]:
import torchvision, torchvision.transforms as T

model_mnist = nn.Sequential(
    nn.Flatten(),
    nn.Linear(784, 256), nn.ReLU(),
    nn.Linear(256, 128), nn.ReLU(),
    nn.Linear(128, 10)
)
model_mnist.load_state_dict(torch.load('mnist_pt.pth', map_location='cpu'))
model_mnist.eval()

# Stáhni testovací data
transform = T.Compose([T.ToTensor(), T.Normalize((0.1307,), (0.3081,))])
test_ds = torchvision.datasets.MNIST('.', train=False, download=True, transform=transform)
test_loader = torch.utils.data.DataLoader(test_ds, batch_size=1000)

# Přesnost
correct = 0
with torch.no_grad():
    for X, y in test_loader:
        correct += (model_mnist(X).argmax(1) == y).sum().item()
print(f'Testovací přesnost: {correct/len(test_ds)*100:.2f}%')

In [ ]:
# Ukázka 16 predikcí
images, labels = next(iter(torch.utils.data.DataLoader(test_ds, batch_size=16, shuffle=True)))
with torch.no_grad():
    preds = model_mnist(images).argmax(1)

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(images[i].squeeze(), cmap='gray')
    ok = preds[i] == labels[i]
    ax.set_title(f'{preds[i]}', color='green' if ok else 'red', fontsize=11)
    ax.axis('off')
plt.suptitle('Predikce (zelená = správně, červená = špatně)', fontsize=12)
plt.tight_layout(); plt.show()

---
## Natrénuj si vlastní model

Chceš model přetrénovat od nuly? Stačí odkomentovat a spustit:

In [ ]:
# # Trénink MNIST od nuly (cca 5 min na CPU, 30s na GPU)
# model_new = nn.Sequential(
#     nn.Flatten(),
#     nn.Linear(784, 256), nn.ReLU(),
#     nn.Linear(256, 128), nn.ReLU(),
#     nn.Linear(128, 10)
# )
# optimizer = torch.optim.SGD(model_new.parameters(), lr=0.1, momentum=0.9)
# train_loader = torch.utils.data.DataLoader(
#     torchvision.datasets.MNIST('.', train=True, download=True, transform=transform),
#     batch_size=128, shuffle=True)
#
# for epoch in range(20):
#     for X, y in train_loader:
#         loss = nn.CrossEntropyLoss()(model_new(X), y)
#         optimizer.zero_grad(); loss.backward(); optimizer.step()
#     print(f'Epocha {epoch+1}/20')
#
# torch.save(model_new.state_dict(), 'muj_mnist.pth')
print('Odkomentuj kód výše a spusť buňku pro trénink od nuly.')